In [23]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [24]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [25]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [26]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [27]:
train_data.sample(10)      # we have some HTML tags,extra spaces,special characters in data which we have to remove
# we have to clean and preprocess the data

,id,dialogue,summary
3092,13729389,Liz: yoooo i saw the cutest movie yesterday <3...,"Liz saw the movie ""Dumplin'""and recommends it ..."
8301,13729203,"Rick: Hi darling, you alright?\r\nJodi: Yes, t...",Jodi is busy around the babies she takes care ...
13,13729567,Leon: did you find the job yet?\r\nArthur: no ...,Arthur is still unemployed. Leon sends him a j...
1149,13815549,Gene: Hi sorry to get back to you so late\r\nG...,Gene and Drew will talk about the video after ...
8519,13730923,"Monica: Ross, do you have an Umbrella??\r\nRos...",Ross has an umbrella. Monica will borrow an um...
9121,13729737,Mona: I'm going to make my first pizza ever\r\...,Tina instructed Mona how to make a pizza. It w...
6151,13611800,Stacy: Please review my edits on your press re...,Stacy edited Doug's press release and needs hi...
8008,13728494,David: Hi \r\nJon: Hi\r\nDavid: What's up \r\n...,Jon is at home watching tv. He's tired and not...
13687,13813986,Bella: I just asked my father to take me to pa...,Bella will ask her mother to take her to the p...
405,13729847,Joanna: What recipe did you use yesterday? The...,Monica has sent Joanna the chicken recipe she ...


In [28]:
train_data.shape

(14732, 3)

In [29]:
val_data.shape

(818, 3)

In [30]:
# random sampling
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [31]:
train_data.shape

(4000, 3)

# Data pre-processing

In [32]:
# When we deal with NLP tasks, text, based on the patterns of text if we want to clean the text, we have the important module Regex(regular expression)

In [33]:
import re

def clean_data(text):
    text=re.sub(r"\r\n"," ",text) # replacing lines
    text=re.sub(r"\s+"," ",text) # removing spaces
    text=re.sub(r"<.*?>"," ",text) # removing HTML tags
    text=text.strip().lower() # removing the trailing spaces and covert all text into lower case
    return text

In [34]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)

In [35]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenize

In [36]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [37]:
# raw data => tokenized inputs for fine-tuning (training model)
def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    target=tokenizer(data["dialogue"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"]=target["input_ids"] # token ids => add to input as labels
    return inputs

In [38]:
train_dataset=train_data.apply(tokenize,axis=1).tolist() # .tolist() => converting into list, because it is compatible with the hugging face transformer
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [39]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [40]:
# input ids - dialogue => token ids

# 1 => EOS (end of sequence) , 0 => padding

# attention mask - it tells us which are valid values and which are padding values, in this wherever it is 1 => valid token id available, wherever it is 0 => invalid/padding values which we don't need to consider during training

# labels - target => summary token

In [41]:
len(train_dataset[0]["input_ids"])

512

In [42]:
type(train_dataset)
type(val_dataset)

list

# working with our model

In [43]:
# text summarization - it is a generational NLP task (GenAI based task), this generation is conditional -> dependent on input
model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [44]:
# fine-tune => performing a specific task for our particular application
import torch
if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

print("device :",device)
model.to(device)

device : cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [45]:
# Training Arguments

training_args=TrainingArguments(
    output_dir="./results",
    
    num_train_epochs=6,
    weight_decay=0.01,
    
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    
    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 -> lr default
)
    

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [ ]:
import torch
print("Is GPU available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))


In [ ]:
# train the model
trainer.train()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\arnav\.conda\envs\deeplearning\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# model load => fine-tune => save the model

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

## Test the Core Logic for Summarization

In [ ]:
def summarize_dialogue(dialogue):
    dialogue=clean_data(dialogue) # Clean

    # Tokenize
    inputs=tokenizer(
        dialogue,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summery => token ids
    model.to(device)
    targets=model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
        max_length=150,
        num_beams=4,
        early_stopping=True
    )
    
    
    # token ids convert to summary => decoding the output
    summary=tokenizer.decode(targets[0],skip_special_tokens=True)  # EOS, SEP
    return summary

In [ ]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)